# Selecting Cavity Walls

## Overview
In this example, we look at how to select the walls of an arbitrarily shaped cavity in mofography. 

This methodology is very useful:
- For computing the stress on a wall, and you need to define a plane or a mesh or select atoms close to a wall
- The cage wall doesn't have to be flat
- It doesn't have to align with any common cell directions (a, b, c, ab, bc, ca, abc etc.)

This may get complicated
- The shape of the cage may not be well defined in some materials. May not be able to break it down into distinct walls
- Picking atoms near a wall using the cell list sometimes causes segmentation fault if no bonds can be detected, in this case pick the atoms and compute bonds manually without the cell list.


## How ? 

This is a simple mesh segmentation using the coordinates and normals of parts of isosurface mesh. This is the same methodology as detecting the limbs vs the body vs head of a 3D mesh of an animal.

## Imports

Import mofography and other modules needed to get started


Change renderer to 'notebook_connected' or 'browser' for interactive plots

In [ ]:
import sys
import os
# * Put your path here
path_to_mgr = os.path.abspath(
    'c:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(path_to_mgr)
import mofography as mgr


import plotly.io as pio 
pio.renderers.default='notebook_connected'

path_to_cif = os.path.join(path_to_mgr, 'examples',
                           'TestCIFs', 'P1-IRMOF-1.cif')


## Compute region segmentation

Same steps as in the region segmentation tutorial in the main repo

In [ ]:
from ase.io import read
mat_atoms = read(path_to_cif)
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
dgrid = mgr.dgrid_from_atoms(mat_updated, spacing=0.25)
regions, maxima = mgr.regions_from_dgrid(dgrid.compute(), mask_thickness=1.0)


## Create isosurface mesh and Select Cavity

We carve out an isosurface mesh and then select the cages of insterest using their region labels. For example if we visualize what we have so far, and say we want to select the cavity numbered in 'select'

Here's all the cages visualized:

### Visualize

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

cscale = px.colors.qualitative.Plotly

isosurface_trace = mgr.plot_surface_dgrid(dgrid.compute(
), isovalue=2.5, ase_atoms=mat_updated, region_labels=regions, cscale=cscale, opacity=1, stylize=True)
cell_trace = mgr.plot_cell(mat_updated)
fig = go.Figure(data=[isosurface_trace]).update(layout=mgr.white_layout)
fig.show(renderer='png')
# frame_trace = mgr.plot_framework_as_mesh(mat_updated)


### Create submesh

We are going to use the region labels to select the vertices that form cavity numberd in select from the overall isosurface

In [ ]:
from scipy.interpolate import RegularGridInterpolator as RGI
import trimesh, numpy as np
from ase.geometry import complete_cell

# select this regions
select=[18] 

# cell matrix
cell = complete_cell(mat_atoms.get_cell())

# isosurfae of all cavities at 2 angstroms
isomesh = mgr.get_isosurface_mesh(dgrid.compute(), 2,cell)

# vertices and faces of the isosurface in fractional coordinates
verts, faces = isomesh.vertices, isomesh.faces
verts = mgr.get_fractional_coordinates(verts, mat_atoms)

# interpolator for region labels
rl = mgr.interpolate_labels(regions)

# pick faces for which the region labels match
select_these_faces = np.where(np.all(np.isin(rl(verts[faces]), select, assume_unique=False), axis=1))

# make a submesh using the selected faces
submesh = isomesh.submesh(select_these_faces, append=True).subdivide()


### Visualize submesh

Let's plot and see if we have the correct submesh. 

In [ ]:
cell_trace = mgr.plot_cell(mat_updated)
submesh_trace = mgr.plot_mesh(
    [submesh], stylize=True, name='submesh for regions {}'.format(select))
go.Figure(data=[cell_trace, submesh_trace]).update(layout=mgr.white_layout).show(renderer='png')


this cage looks like a truncated octahedron, where the faces are curved instead of flat triagles. It has 12 faces, and 6 windows. We want to detect and distinguish the 12 faces.

### Segment Mesh

We are going to use agglomerative  clustering to group the vertices of the above mesh into different faces. At each vertex, we have 6 values (features), 3 positions and 3 direction cosines for the normal.

Note: We can use PCA, t-SNE, UMAP etc. to reduce the dimensionality, but here we need not, since we have only 6 features.

In [ ]:
# Create a scaled set of features for better clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering 
X = np.hstack((submesh.vertices, submesh.vertex_normals))
X = StandardScaler().fit_transform(X)

# # Use PCA
# from sklearn.decomposition import PCA
# X_pca = PCA(n_components=5).fit_transform(X_scaled)


# from sklearn.manifold import TSNE 
# X = TSNE(n_components=3).fit_transform(X)

# agglomerative clustering, the distance threshold doesn't matter that much.
clst = AgglomerativeClustering(linkage='single',distance_threshold=0.25, n_clusters=None)
clst.fit(X)
labels = clst.labels_


Note:
-------
As we will see shortly, one natural drawback is the clustering algorithm gets confused at the ridges between two faces, since the normals can fluctuate a lot close to the edge where they become discontinuous. But, we can easily eliminate these ridge pixels, since they will few in number in between.

### Eliminate Ridge clusters

These clusters will have far too few points to be a face. Let's elimiate them. 

In [ ]:

# counts in each cluster
_, counts = np.unique(labels, return_counts=True)

# remove clusters with les than 100 points,
counts_cutoff=100
keep_these_labels = np.unique(labels)[np.where(np.array(counts)>counts_cutoff)]


# grab vertices corresponding to valid clusters
vertex_selections = [submesh.vertices[np.where(labels==l)] for l in keep_these_labels]

# get centroids for valid clusters
cage_wall_centroids = np.array([np.mean(s,axis=0) for s in vertex_selections])


## Visualize clustering

Let's visualize the results of our clustering.


### Relabel
But first let's relabel the vertices from 0 to n-1, where n is the number of valid clusters. All the points belonging to invalid clusters are labeled -1. We are going to call these labels unique_colorvals, as we will be using them for plotting

In [ ]:
# relabel as described above
colorvals = [l if l in keep_these_labels else -1 for l in labels]
_,unique_colorvals = np.unique(colorvals, return_inverse=True)



### Plot the results

Now we can apply a colorscale using matplotlib and visualize the results using mofography (plotly, basically).

In [ ]:


# apply colormap to get a list of colors and add that to the mesh
import matplotlib as mpl 
colors = mpl.cm.get_cmap('tab20c')(unique_colorvals)
colors[:, :-1] = (colors[:, :-1]*255).astype(int)
submesh.visual.vertex_colors=colors
submesh_trace = mgr.plot_mesh(
    [submesh], hovertext=np.array(unique_colorvals).astype(str), stylize=True, name='submesh for region {}'.format(select[0]))
go.Figure(data=submesh_trace).update(layout=mgr.white_layout).show(renderer='png)


## Pick framework atoms

Now that we have the centroids of the walls, we can pick atoms from the unit cell that are close to one or more of the centroids very easily using the AABB cell list from freud that is already implementd in mofography. In this example we are simply selecting all atoms within 'r_max' distance using 'ball' query, instead you may also choose n_neighbors using a 'nearest' query.

In [ ]:
aabb=mgr.AABB_on_atoms(mat_atoms)
ab_r = aabb.query(cage_wall_centroids, query_args=dict(mode='ball',  r_max=5))
ab_r_nl = ab_r.toNeighborList(sort_by_distance=True)
pick_these_atoms = ab_r_nl.point_indices 

In [ ]:
pick_these_atoms

### Plot the results

Let's quickly add traces for the centroids of the walls, and the subset of frame atoms and plot everything for confirmation

In [ ]:
centroids_trace = mgr.scatter3d(cage_wall_centroids, name='centroids of walls')
subset_atoms_trace = mgr.plot_subset_of_atoms(mat_updated,pick_these_atoms )


In [ ]:
fig_final = go.Figure(data=[cell_trace, submesh_trace, centroids_trace, *subset_atoms_trace]).update(layout=mgr.white_layout)
# fig_final = go.Figure(data=[submesh_trace, centroids_trace]).update(layout=mgr.white_layout)
fig_final.show(renderer='notebook_connected')

In [ ]:
# fig_final.write_html('one_cage_wall_segmentation.html')